# Rent 06 · Clean HPD violations
**Input:** `data/raw/hpd_violations/` · **Output:** `data/processed/hpd_violations_clean.parquet`

Steps:
1. Drop duplicate ids.
2. Parse dates (2022+ inspections).
3. Validate the BBL.
4. Keep classes A/B/C/I.
5. **Tag the free-text description with regex** into topics, then map the topics onto the same problem types as complaints.
6. Add flags: open, lead paint, rent-impairing.

In [1]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config, housing as H, clean_housing as C
from src.preprocess import CleaningReport

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_colwidth", 70)

def show_report(rep):
    return pd.DataFrame(rep.steps)[["step", "rows_before", "rows_after", "rows_removed", "detail"]]

def save(df, name, rep):
    df.to_parquet(C.OUT[name], index=False)
    (config.REPORTS_DIR / f"cleaning_report_{name}.json").write_text(json.dumps(rep.to_dict(), indent=2, default=str))
    print(f"Saved {len(df):,} rows × {df.shape[1]} columns → {C.OUT[name].relative_to(ROOT)}")

In [2]:
raw = H.load("hpd_violations")
rep = CleaningReport("hpd_violations")
clean = C.clean_violations(raw, rep)
show_report(rep)

,step,rows_before,rows_after,rows_removed,detail
0,load_raw,4492764,4492764,0,
1,drop_duplicate_violationid,4492764,4492764,0,
2,parse_dates,4492764,4492764,0,missing or pre-2022 inspectiondate
3,valid_bbl,4492764,4488073,4691,missing or invalid BBL
4,valid_class,4488073,4488073,0,class not in A/B/C/I
5,derive_fields,4488073,4488073,0,"topic via regex, problem_type, open/lead/rent-impairing flags"


### Regex tagging: the patterns, and how much they cover

In [3]:
display(pd.Series(H.VIOLATION_TOPICS, name="regex pattern").to_frame())
print(f"Tagged with a topic: {(clean['topic'] != 'other').mean():.1%}")
pd.crosstab(clean["problem_type"], clean["class"], margins=True).sort_values("All", ascending=False)

,regex pattern
heat/hot water,\bHEAT|HOT WATER|BOILER
pests,ROACH|MICE|\bRATS?\b|RODENT|VERMIN|BED ?BUG|INSECT|PEST
mold/leaks,\bMOLD|MOULD|LEAK|WATER DAMAGE|DAMP
lead paint,LEAD
paint/plaster,PAINT|PLASTER
fire safety,SMOKE|CARBON MONOXIDE|FIRE|SPRINKLER|EGRESS
windows/guards,WINDOW
doors/locks,DOOR|LOCK
electric,ELECTRIC|OUTLET|WIRING|LIGHT
plumbing,PLUMB|TOILET|SINK|FAUCET|DRAIN|BASIN|BATH


Tagged with a topic: 92.1%


class,A,B,C,I,All
problem_type,,,,,
All,1107723,1714231,1269275,396844,4488073
paint/plaster,302171,334414,180338,1,816924
other,114663,231995,64886,396544,808088
pests,418999,31984,306070,0,757053
doors/windows,44441,184605,363605,272,592923
leaks/mold,128487,313657,47596,0,489740
safety,42562,365115,57045,25,464747
heat/hot water,4536,11033,213489,1,229059
plumbing,38190,155297,21209,0,214696


In [4]:
# Spot-check: three random descriptions per problem type
for pt, g in clean.merge(raw[["violationid", "novdescription"]], on="violationid").groupby("problem_type"):
    print(f"\n[{pt}]")
    for t in g["novdescription"].dropna().sample(min(3, len(g)), random_state=0):
        print("   ", str(t)[:140])


[doors/windows]
    § 27-2005, 27-2007, 27-2041.1 HMC, §238, § 309; § 107 (2) ( C) MDL AND 28 RCNY §25-171: REPLACE OR REPAIR THE SELF-CLOSING DOORS THAT IS MIS
    § 27-2005, 27-2007, 27-2041.1 HMC, §238, § 309; § 107 (2) ( C) MDL AND 28 RCNY §25-171: REPLACE OR REPAIR THE SELF-CLOSING DOORS THAT IS MIS
    § 27-2005 HMC: PROPERLY REPAIR OR REPLACE THE BROKEN OR DEFECTIVE MORTISE LOCK AT DOOR IN THE ENTRANCE LOCATED AT APT D1, 4th STORY, 3rd APA

[electric]
    § 27-2005, 2006, 2037 HMC: PROPERLY REPAIR AND ABATE UNSAFE ELECTRIC WIRING CONDITION CONSISTING OF EXPOSE ELECTRICAL WIRES AT CEILING IN TH
    § 27-2005 HMC: PROPERLY REPAIR OR REPLACE THE BROKEN OR DEFECTIVE ELECTRICAL OUTLET AT EAST WALL IN THE KITCHEN LOCATED AT APT 3E, 3rd STORY
    § 27-2037, 2038 HMC: PROVIDE A SAFE AND ADEQUATE SUPPLY OF ELECTRIC SERVICE TO THE FIXTURES AT ELECTRICAL LIIGHT FIXTURE AT CEILING IN THE 1

[heat/hot water]
    § 27-2005, 27-2007, 27-2041.1 HMC, §238, § 309; § 107 (2) ( C) MDL AND 28 RCNY 

    §27-2107 ADM CODE OWNER FAILED TO FILE A VALID REGISTRATION STATEMENT WITH THE DEPARTMENT AS REQUIRED BY ADM CODE §27-2097 AND IS THEREFORE 
    § 27-2005 ADM CODE PROPERLY REPAIR WITH SIMILAR MATERIAL THE BROKEN OR DEFECTIVE ERODED BRICK AND MORTAR AT BUILDING FACADE FROM PARAPET DOW
    §27-2107 ADM CODE OWNER FAILED TO FILE A VALID REGISTRATION STATEMENT WITH THE DEPARTMENT AS REQUIRED BY ADM CODE §27-2097 AND IS THEREFORE 

[paint/plaster]
    § 27-2005 ADM CODE REPAIR THE BROKEN OR DEFECTIVE PLASTERED SURFACES AND PAINT IN A UNIFORM COLOR AT CEILING IN THE KITCHEN LOCATED AT APT 5
    § 27-2005 ADM CODE REPAIR THE BROKEN OR DEFECTIVE PLASTERED SURFACES AND PAINT IN A UNIFORM COLOR AT HEADER OF 2ND WINDOW FROM NORTH AT EAST
    § 27-2005 ADM CODE REPAIR THE BROKEN OR DEFECTIVE PLASTERED SURFACES AND PAINT IN A UNIFORM COLOR THE CEILING AND EAST WALL IN THE 2nd PRIVA

[pests]
    (A) § HMC:FILE ANNUAL BEDBUG REPORT IN ACCORDANCE WITH HPD RULE AS DESCRIBED ON THE BACK OF THIS NOT

In [5]:
print(f"Open: {clean['is_open'].mean():.1%} | class C: {(clean['class'] == 'C').mean():.1%} | "
      f"lead: {clean['is_lead'].mean():.1%} | rent-impairing: {clean['is_rent_impairing'].mean():.1%}")
save(clean, "hpd_violations", rep)

Open: 34.0% | class C: 28.3% | lead: 3.9% | rent-impairing: 6.7%


Saved 4,488,073 rows × 14 columns → data/processed/hpd_violations_clean.parquet
